    # 정규 표현식을 이용한 텍스트 탐색과 정제

정규 표현식은 문자열의 반복·범위·경계 규칙을 하나의 패턴으로 표현해 원하는 부분을 찾거나 바꾸는 도구이다. 개인정보 마스킹, 로그 분석, 입력 검증과 토큰화 전 정제에 사용한다.

- **정규 표현식(Regular Expression)**: 찾으려는 문자열 규칙을 기호로 표현한 패턴
- **`re` 모듈**: 정규 표현식의 탐색·추출·치환 기능을 제공하는 Python 표준 라이브러리
- **매칭(Matching)**: 문자열의 일부가 패턴 규칙과 일치하는지 판별하는 과정


In [2]:
import re


## 기본문법

정규 표현식은 일반 문자를 그대로 찾는 규칙과 특별한 기능을 가진 기호를 조합해 작성한다. 기호의 의미를 먼저 익힌 뒤 성공 사례와 실패 사례를 함께 비교한다.

- **리터럴(Literal)**: `a`, `7`처럼 자기 자신과 일치하는 일반 문자
- **메타 문자(Metacharacter)**: `.`, `*`, `[]`처럼 특별한 매칭 규칙을 나타내는 문자
- **원시 문자열(Raw String)**: `r'\d+'`처럼 역슬래시를 Python 이스케이프가 아닌 정규식 기호로 전달하기 쉬운 문자열 표기
- **Match 객체**: 일치 문자열과 위치 범위 `span`을 담고, 불일치 시 `None`을 반환하는 결과 객체


### `.`과 문자열 탐색

문자 한 자리는 무엇이 와도 되지만 앞뒤 문자는 고정해야 할 때 점을 사용한다.

- **`.`**: 기본 설정에서 줄바꿈을 제외한 임의의 문자 한 개와 일치
- **`re.compile()`**: 패턴 문자열을 재사용할 수 있는 `Pattern` 객체로 만드는 함수
- **`Pattern.search()`**: 문자열 전체에서 첫 번째 일치 구간을 찾는 메서드

`a.c`는 `abc`, `a8c`, `a c`를 찾지만 가운데 문자가 없는 `ac`에서는 `None`을 반환한다. 마지막 결과의 `span=(7, 10)`은 문자열 중간에서도 탐색한다는 뜻이다.


In [3]:
pattern = re.compile('a.c')
print(type(pattern), pattern)

print(pattern.search('abc'))
print(pattern.search('a8c'))
print(pattern.search('a c'))
print(pattern.search('ac'))
print(pattern.search('I love abc!'))


<class 're.Pattern'> re.compile('a.c')
<re.Match object; span=(0, 3), match='abc'>
<re.Match object; span=(0, 3), match='a8c'>
<re.Match object; span=(0, 3), match='a c'>
None
<re.Match object; span=(7, 10), match='abc'>


### `?` 수량자

문자가 반드시 있지는 않지만 최대 한 번만 나타나는 형식을 표현할 때 사용한다.

- **`?`**: 바로 앞의 문자나 그룹이 0번 또는 1번 나타나는 경우와 일치

`ab?c`는 `abc`와 `ac`를 찾지만 `b`가 두 개인 `abbc`는 찾지 못한다.


In [4]:
pattern = re.compile('ab?c')

print(pattern.search('abc'))
print(pattern.search('ac'))
print(pattern.search('abbc'))


<re.Match object; span=(0, 3), match='abc'>
<re.Match object; span=(0, 2), match='ac'>
None


### `*` 수량자

같은 문자가 없을 수도 있고 여러 번 반복될 수도 있는 부분을 표현한다.

- **`*`**: 바로 앞의 문자나 그룹이 0번 이상 반복되는 경우와 일치

`ab*c`는 `ac`부터 `abbbbbbbbbc`까지 찾지만 `b` 대신 `X`가 있는 `aXc`는 찾지 못한다.


In [5]:
pattern = re.compile('ab*c')

print(pattern.search('ac'))
print(pattern.search('abc'))
print(pattern.search('abbbbbbbbbc'))

print(pattern.search('aXc'))


<re.Match object; span=(0, 2), match='ac'>
<re.Match object; span=(0, 3), match='abc'>
<re.Match object; span=(0, 11), match='abbbbbbbbbc'>
None


### `.*`와 탐욕적 매칭

시작과 끝만 정하고 사이의 길이를 제한하지 않을 때 `.`과 `*`를 결합한다.

- **탐욕적 매칭(Greedy Matching)**: 수량자가 조건을 만족하는 범위에서 가능한 한 긴 문자열을 선택하는 기본 방식
- **게으른 매칭(Lazy Matching)**: 수량자 뒤에 `?`를 붙여 가능한 한 짧은 문자열을 선택하는 방식

`a.*c`는 현재 예제의 `a`부터 마지막 `c`까지 찾는다. 예를 들어 `a1c2c`에서 `a.*c`는 전체를, `a.*?c`는 `a1c`까지만 선택한다.


In [6]:
pattern = re.compile('a.*c')

print(pattern.search('aㅋㅋㅋㅋc'))
print(pattern.search('ac'))
print(pattern.search('a123456789c'))
print(pattern.search('aXc'))


<re.Match object; span=(0, 6), match='aㅋㅋㅋㅋc'>
<re.Match object; span=(0, 2), match='ac'>
<re.Match object; span=(0, 11), match='a123456789c'>
<re.Match object; span=(0, 3), match='aXc'>


### `+` 수량자

문자가 한 번 이상 반드시 등장해야 하는 반복 규칙에는 `+`를 사용한다.

- **`+`**: 바로 앞의 문자나 그룹이 1번 이상 반복되는 경우와 일치

`ab+c`는 `abc`와 여러 개의 `b`를 포함한 문자열을 찾지만 `b`가 없는 `ac`에서는 `None`을 반환한다.


In [7]:
pattern = re.compile('ab+c')

print(pattern.search('abc'))
print(pattern.search('abbbbbbbbbc'))
print(pattern.search('ac'))


<re.Match object; span=(0, 3), match='abc'>
<re.Match object; span=(0, 11), match='abbbbbbbbbc'>
None


### 중괄호 수량자

반복 횟수의 하한과 상한을 명확히 제한해야 하는 코드, 번호와 날짜 형식에 사용한다.

- **`{n}`**: 바로 앞의 문자나 그룹이 정확히 `n`번 반복되는 경우와 일치
- **`{min,max}`**: 반복 횟수가 `min` 이상 `max` 이하인 경우와 일치
- **`{min,}`**: 반복 횟수가 `min` 이상인 경우와 일치

첫 실습의 `ab{2}c`는 `b`가 정확히 두 개인 `abbc`만 찾는지 확인한다.


In [8]:
pattern = re.compile('ab{2}c')

print(pattern.search('abbc'))
print(pattern.search('abc'))
print(pattern.search('abbbbbbbbbc'))
print(pattern.search('ac'))


<re.Match object; span=(0, 4), match='abbc'>
None
None
None


### 반복 횟수 범위 지정

정확한 횟수 대신 허용 범위를 지정하면 길이가 조금씩 다른 형식을 하나의 패턴으로 처리할 수 있다.

- **`{1,3}`**: 바로 앞의 요소가 1번 이상 3번 이하 반복되는 경우와 일치

`ab{1,3}c`는 `abc`, `abbc`, `abbbc`를 찾고 범위를 벗어난 `ac`와 `abbbbbbbbbc`는 찾지 못한다.


In [9]:
pattern = re.compile('ab{1,3}c')

print(pattern.search('ac'))
print(pattern.search('abc'))
print(pattern.search('abbc'))
print(pattern.search('abbbc'))
print(pattern.search('abbbbbbbbbc'))


None
<re.Match object; span=(0, 3), match='abc'>
<re.Match object; span=(0, 4), match='abbc'>
<re.Match object; span=(0, 5), match='abbbc'>
None


### 축약 수량자와 중괄호 수량자 비교

짧은 기호와 명시적인 반복 범위는 표현만 다를 뿐 같은 규칙을 만들 수 있다.

- **`?` / `{0,1}`**: 0번 또는 1번 반복을 나타내는 같은 규칙
- **`*` / `{0,}`**: 0번 이상 반복을 나타내는 같은 규칙
- **`+` / `{1,}`**: 1번 이상 반복을 나타내는 같은 규칙
- **`re.fullmatch()`**: 문자열 일부가 아닌 전체가 패턴과 일치하는지 검사하는 함수

저장 출력의 각 패턴 쌍에서 왼쪽과 오른쪽의 `True`·`False` 목록이 모두 같은지 확인한다.


In [10]:
sample_texts = ['ac', 'abc', 'abbc', 'abbbc']

equivalent_patterns = {
    '? 와 {0,1}': (r'ab?c', r'ab{0,1}c'),
    '* 와 {0,}': (r'ab*c', r'ab{0,}c'),
    '+ 와 {1,}': (r'ab+c', r'ab{1,}c'),
}

for description, (short_pattern, range_pattern) in equivalent_patterns.items():
    short_results = [bool(re.fullmatch(short_pattern, text)) for text in sample_texts]
    range_results = [bool(re.fullmatch(range_pattern, text)) for text in sample_texts]
    print(f'{description}: {short_results} == {range_results}')


? 와 {0,1}: [True, True, False, False] == [True, True, False, False]
* 와 {0,}: [True, True, True, True] == [True, True, True, True]
+ 와 {1,}: [False, True, True, True] == [False, True, True, True]


### `[]` 문자 클래스

한 위치에 올 수 있는 문자의 목록이나 범위를 묶으면 문자마다 OR 조건을 반복하지 않아도 된다.

- **문자 클래스(Character Class)**: `[]` 안의 문자 또는 범위 중 한 문자와 일치하는 규칙
- **범위 `-`**: `[a-z]`, `[가-힣]`처럼 연속된 문자 범위를 표현
- **부정 문자 클래스 `[^...]`**: 대괄호 첫 위치의 `^`를 사용해 지정한 문자 이외의 한 문자와 일치
- **`re.findall()`**: 문자열에서 겹치지 않는 모든 일치 결과를 리스트로 반환하는 함수

출력에서 `[A-Za-z]`는 영문자만 찾지만 `[A-z]`는 문자 코드 범위에 포함된 `^`까지 찾는다. 한글도 `[가-힣]`과 `[가-힣ㄱ-ㅎㅏ-ㅣ]`의 결과 차이를 비교한다.


In [11]:
src = "ABCDEFGhijklmn abc 가나다라마바사ㅋㅋㅎㅎㅓㅓㅏㅏ 1234567890 !@#$%^&*() \t\n"

print(re.findall(r'[aj]', src))
print(re.findall(r'[0123456789]', src))
print(re.findall(r'[a-z]', src))
print(re.findall(r'[A-Z]', src))
print(re.findall(r'[A-Za-z]', src))
print(re.findall(r'[A-z]', src))

print(re.findall(r'[가-힣]', src))
print(re.findall(r'[가-힣ㄱ-ㅎㅏ-ㅣ]', src))

# [^]: Not
print(re.findall(r'[^가-힣]', src))


['j', 'a']
['1', '2', '3', '4', '5', '6', '7', '8', '9', '0']
['h', 'i', 'j', 'k', 'l', 'm', 'n', 'a', 'b', 'c']
['A', 'B', 'C', 'D', 'E', 'F', 'G']
['A', 'B', 'C', 'D', 'E', 'F', 'G', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'a', 'b', 'c']
['A', 'B', 'C', 'D', 'E', 'F', 'G', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'a', 'b', 'c', '^']
['가', '나', '다', '라', '마', '바', '사']
['가', '나', '다', '라', '마', '바', '사', 'ㅋ', 'ㅋ', 'ㅎ', 'ㅎ', 'ㅓ', 'ㅓ', 'ㅏ', 'ㅏ']
['A', 'B', 'C', 'D', 'E', 'F', 'G', 'h', 'i', 'j', 'k', 'l', 'm', 'n', ' ', 'a', 'b', 'c', ' ', 'ㅋ', 'ㅋ', 'ㅎ', 'ㅎ', 'ㅓ', 'ㅓ', 'ㅏ', 'ㅏ', ' ', '1', '2', '3', '4', '5', '6', '7', '8', '9', '0', ' ', '!', '@', '#', '$', '%', '^', '&', '*', '(', ')', ' ', '\t', '\n']


### 숫자 범위와 반복 횟수

연속된 숫자의 자릿수를 제한하면 인증 번호나 코드 조각처럼 고정 길이 형식을 찾을 수 있다.

- **`[0-9]`**: 숫자 한 자리와 일치하는 문자 클래스
- **`[0-9]{3}`**: 공백 없이 이어진 숫자 세 자리와 일치

출력에서 `123`은 매칭되고 `12 34 56`은 `None`이 된다. `123456789`에서는 `search()`가 처음 찾은 세 자리 `123`만 반환한다.


In [12]:
pattern = re.compile('[0-9]{3}')

print(pattern.search('안녕하세요~ 저는 123입니다.'))
print(pattern.search('12 34 56'))
print(pattern.search('123456789'))


<re.Match object; span=(10, 13), match='123'>
None
<re.Match object; span=(0, 3), match='123'>


### 전화번호 추출과 마스킹

같은 전화번호 패턴을 탐색에 사용하면 번호를 추출하고, 치환에 사용하면 개인정보를 가릴 수 있다.

- **`Match.group()`**: Match 객체에서 실제로 일치한 문자열을 반환하는 메서드
- **`Pattern.sub(repl, text)`**: 패턴과 일치하는 부분을 `repl` 문자열로 바꾸는 메서드

두 형식의 전화번호가 각각 추출되고, 마지막 문장에서는 `010-1234-5678`이 별표 3개-4개-4개 형식으로 바뀌는지 확인한다.


In [13]:
# [0-9]{3}-[0-9]{3,4}-[0-9]{4}

phone_pattern = re.compile(r'[0-9]{3}-[0-9]{3,4}-[0-9]{4}')
phone_texts = ['제 번호는 010-1234-5678입니다.', '☎️콜센터☎️: 070-123-4567']

for phone_text in phone_texts:

    match = phone_pattern.search(phone_text)

    # 매칭되면 group()으로 전화번호 문자열만 출력
    print(match.group() if match else None)

private_text = '제 번호는 010-1234-5678입니다.'
# 찾은 전화번호를 동일한 마스킹 문자열로 치환
masked_text = phone_pattern.sub('***-****-****', private_text)
print('마스킹 결과:', masked_text)

010-1234-5678
070-123-4567
마스킹 결과: 제 번호는 ***-****-****입니다.


### 기본 문자 클래스

자주 사용하는 문자 범위는 역슬래시 문자 클래스로 짧게 작성할 수 있다. Python 정규식의 `\w`와 `\d`는 기본적으로 유니코드 문자를 인식한다.

- **`\w`**: 영문자·숫자·밑줄과 한글을 포함한 유니코드 단어 문자와 일치
- **`\d`**: 유니코드 십진 숫자와 일치
- **`\s`**: 공백, 탭과 줄바꿈 같은 공백 문자와 일치

출력에서 `ID_007은`이 하나의 `\w+` 결과로 유지되고, `\d+`는 `007` 두 개를 추출하며 `\s+`는 줄바꿈까지 찾는지 확인한다.


In [14]:
text = 'ID_007 is available.\nID_007은 사용가능합니다.'

print(re.findall(r'\w+', text))
print(re.findall(r'[\w가-힣]+', text))
print(re.findall(r'\d+', text))
print(re.findall(r'\s+', text))


['ID_007', 'is', 'available', 'ID_007은', '사용가능합니다']
['ID_007', 'is', 'available', 'ID_007은', '사용가능합니다']
['007', '007']
[' ', ' ', '\n', ' ']


### 반대 문자 클래스

소문자 문자 클래스가 포함 대상을 정한다면 대응하는 대문자 클래스는 그 범위에 포함되지 않는 대상을 찾는다.

- **`\W`**: `\w`에 포함되지 않는 문자를 탐색
- **`\D`**: `\d`에 포함되지 않는 문자를 탐색
- **`\S`**: `\s`에 포함되지 않는 문자를 탐색

출력에서 `\W+`는 공백과 구두점 묶음을, `\S+`는 공백으로 나뉜 문자열 묶음을 반환하는지 비교한다.


In [15]:
print(re.findall(r'\W+', text))
print(re.findall(r'\D+', text))
print(re.findall(r'\S+', text))


[' ', ' ', '.\n', ' ', '.']
['ID_', ' is available.\nID_', '은 사용가능합니다.']
['ID_007', 'is', 'available.', 'ID_007은', '사용가능합니다.']


### `^`, `$` 앵커와 경계

문자열의 내용뿐 아니라 시작과 끝 위치까지 제한해야 입력 형식 전체를 더 정확히 검사할 수 있다.

- **앵커(Anchor)**: 문자를 소비하지 않고 문자열의 시작이나 끝 위치를 검사하는 경계 기호
- **`^`**: 기본 모드에서 전체 문자열의 시작 위치와 일치
- **`$`**: 기본 모드에서 전체 문자열의 끝 위치와 일치
- **`re.MULTILINE`**: `^`와 `$`가 각 줄의 시작과 끝도 검사하도록 바꾸는 플래그

`abc`가 처음에 있을 때만 기본 `^abc`가 매칭되고, 줄바꿈 뒤의 `abc`는 `MULTILINE`을 적용한 경우에만 매칭된다.


In [16]:
pattern = re.compile('^abc')

print(pattern.search('abcㅋㅋㅋ'))
print(pattern.search('ㅋㅋㅋabc'))
print(re.search('^abc', 'ㅋㅋㅋ\nabc'))

# re.MULTIILINE: 줄 마다 패턴 찾기 수행하는 플래그
print(re.search('^abc', 'ㅋㅋㅋ\nabc', re.MULTILINE))


<re.Match object; span=(0, 3), match='abc'>
None
None
<re.Match object; span=(4, 7), match='abc'>


### 문자열 끝 탐색

문자열이 특정 단어로 끝나는지 확인할 때 `$`를 사용하며, 뒤에 다른 문자가 붙으면 매칭되지 않는다.

`world$`는 `Hello world`를 찾지만 느낌표가 붙은 문자열과 첫 줄의 `world`는 기본 모드에서 찾지 못한다. `MULTILINE`을 적용하면 첫 줄 끝의 `world`도 찾는다.


In [17]:
print(re.search('world$', 'Hello world'))
print(re.search('world$', 'Hello world!!!'))
print(re.search('world$', 'world\ncone'))
print(re.search('world$', 'world\ncone', re.MULTILINE))


<re.Match object; span=(6, 11), match='world'>
None
None
<re.Match object; span=(0, 5), match='world'>


### `|` OR 연산자와 비캡처 그룹

여러 후보 중 하나를 찾을 때 OR 조건을 사용하고, 후보 뒤에 공통 문자열이 이어지면 그룹으로 묶는다.

- **`|`**: 왼쪽 또는 오른쪽 패턴 중 하나와 일치하는 OR 연산자
- **비캡처 그룹 `(?:...)`**: 여러 패턴을 하나로 묶되 캡처 결과에는 저장하지 않는 그룹

첫 출력은 부분 문자열 `on`, `ues`, `rida`를 반환하고, 두 번째 출력은 그룹 뒤의 `day`까지 포함한 세 요일 전체를 반환한다.


In [18]:
pattern = re.compile('on|ues|rida')

text = 'Monday Tuesday Wenesday Thursday Friday Saturday Sunday'

print(re.findall(r'on|ues|rida', text))
print(re.findall(r'(?:Mon|Tues|Fri)day', text))


['on', 'ues', 'rida']
['Monday', 'Tuesday', 'Friday']


### 그룹, 캡처와 역참조

일치한 부분을 치환 결과에서 다시 사용하려면 괄호로 필요한 영역을 캡처한다.

- **그룹(Group) `(...)`**: 여러 패턴을 하나의 단위로 묶는 표현
- **캡처(Capture)**: 그룹과 일치한 문자열을 번호가 있는 결과로 저장
- **역참조(Backreference) `\1`**: 첫 번째 캡처 그룹의 문자열을 패턴이나 치환 문자열에서 다시 사용
- **`re.sub(pattern, repl, text)`**: 모든 일치 부분을 지정한 문자열로 치환하는 함수

`<strong>(.*)</strong>`에서 내부 텍스트만 `\1`로 남겨 출력이 `This is important text`가 되는지 확인한다.


In [20]:
text = "This is <strong>important</strong> text"

pattern = r'<strong>(.*)</strong>'

# '\1'로 치환 == ()로 지정된 챕터 그룹 중 1번을 역참조
print(re.sub(pattern, 'abc', text))

This is abc text


### 같은 태그 이름 확인과 반복 치환

여는 태그의 이름을 캡처하고 닫는 태그에서 다시 참조하면 같은 이름의 태그 쌍을 찾을 수 있다.

- **패턴 안의 `\1`**: 여는 태그와 같은 이름의 닫는 태그를 요구하는 역참조
- **치환 문자열의 `\2`**: 두 번째 캡처 그룹인 태그 내부 문자열만 유지

일치하는 태그가 남아 있는 동안 치환을 반복해 저장 출력이 `Hello World! Visit here.`가 되는지 확인한다.


In [22]:
text = "<div>Hello <b>World</b>! Visit <a href='https://example.com'>here</a>.</div>"

# 시작태그, 종료태그가 쌍으로 존재하는 요쇼 찾기 패턴
pattern = r"<(\w+)[^>]*>(.*)</\1>"

while re.search(pattern, text):
    text = re.sub(pattern, r"\2", text)

print(text)


Hello World! Visit here.


#### HTML에 정규 표현식을 사용할 때의 한계

이 코드는 구조가 단순한 한 겹 태그를 보여 주는 교육용 예제이다. 실제 HTML은 중첩, 속성, 주석, `script`와 깨진 마크업이 섞일 수 있으므로 BeautifulSoup이나 lxml 같은 파서로 구조를 해석하고 정규식은 추출된 텍스트의 후처리에 사용한다.


## 플래그로 매칭 규칙 바꾸기

플래그는 패턴 문자열을 다시 작성하지 않고 대소문자, 줄 경계와 줄바꿈 처리 방식을 바꾼다.

- **`re.IGNORECASE` (`re.I`)**: 영문 대소문자를 구분하지 않는 매칭 설정
- **`re.MULTILINE` (`re.M`)**: `^`와 `$`가 각 줄의 시작과 끝에도 일치하도록 하는 설정
- **`re.DOTALL` (`re.S`)**: 점(`.`)이 줄바꿈 문자까지 포함해 매칭하도록 하는 설정

저장 출력에서 `HELLO`는 두 IGNORECASE 방식 모두 매칭되고, `start\nend`는 DOTALL을 적용한 마지막 경우에만 매칭된다.


In [23]:
pattern = re.compile(r'hello', re.IGNORECASE)
print(pattern.search('HELLO WORLD'))

pattern_str = r'hello'
print(re.search(pattern_str, 'HELLO WORLD', re.IGNORECASE))

multiline_text = 'start\nend'
print(re.search(r'start.*end', multiline_text))
print(re.search(r'start.*end', multiline_text, re.DOTALL))


<re.Match object; span=(0, 5), match='HELLO'>
<re.Match object; span=(0, 5), match='HELLO'>
None
<re.Match object; span=(0, 9), match='start\nend'>


## 정규 표현식 토큰화

정규 표현식을 토큰 경계 규칙으로 사용하면 필요한 문자 묶음만 추출하거나 특정 구분자를 기준으로 문장을 나눌 수 있다.

- **토큰 경계(Token Boundary)**: 하나의 토큰이 시작하고 끝나는 위치를 정하는 규칙
- **`RegexpTokenizer`**: 정규식 패턴으로 토큰을 추출하거나 구분자를 지정하는 NLTK 토크나이저
- **`tokenize()`**: 설정한 패턴을 문자열에 적용해 토큰 리스트를 반환하는 메서드

`[\w]+`는 단어 문자만 추출하므로 아포스트로피와 하이픈 위치가 경계가 되어 `He's`와 `long-distance`가 각각 여러 토큰으로 나뉜다.


In [24]:
from nltk.tokenize import RegexpTokenizer

text = "He's a runner, but not a long-distance runner."

tokenizer = RegexpTokenizer(r'[\w]+')
tokens = tokenizer.tokenize(text)
print(tokens)

['He', 's', 'a', 'runner', 'but', 'not', 'a', 'long', 'distance', 'runner']


### 추출 패턴과 구분자 패턴 비교

앞 실습은 패턴과 일치하는 부분을 토큰으로 추출했지만, 이번에는 공백을 구분자로 사용해 그 사이의 문자열을 토큰으로 남긴다.

- **`gaps=True`**: 패턴과 일치한 부분을 토큰이 아닌 구분자로 해석하는 인자
- **`\s+`**: 하나 이상의 연속된 공백 문자를 구분자로 사용

공백만 제거되므로 출력에서 `He's`, `runner,`, `long-distance`, `runner.`처럼 구두점과 하이픈이 토큰에 남는지 확인한다.


In [25]:
tokenizer = RegexpTokenizer(r'\s+', gaps=True)
tokens = tokenizer.tokenize(text)
print(tokens)


["He's", 'a', 'runner,', 'but', 'not', 'a', 'long-distance', 'runner.']


## 정규 표현식을 이용한 정제

정제에서는 찾은 문자열을 삭제할지, 의미를 나타내는 대체 토큰으로 바꿀지 결정해야 한다. 개인정보는 유형 정보를 남기는 마스킹이 문장 구조 보존에 유리하다.

- **마스킹(Masking)**: 민감한 원문을 직접 노출하지 않으면서 `[EMAIL]` 같은 유형 토큰으로 바꾸는 처리
- **`re.sub()`**: 패턴과 일치하는 모든 부분을 지정한 문자열로 치환하는 함수

이메일의 아이디·도메인·최상위 도메인을 찾는 패턴을 적용해 두 주소가 모두 `[EMAIL]`로 바뀌는지 확인한다.


In [26]:
text = "문의: john.doe@example.com, 회사: contact@company.org"

# 아이디, @, 도메인과 최상위 도메인으로 구성된 이메일 패턴
email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'

# 이메일 주소를 삭제하지 않고 의미를 알 수 있는 [EMAIL] 토큰으로 치환
masked_text = re.sub(email_pattern, '[EMAIL]', text)

print('원문:', text)
print('마스킹:', masked_text)

원문: 문의: john.doe@example.com, 회사: contact@company.org
마스킹: 문의: [EMAIL], 회사: [EMAIL]


### 특수문자 제거

단어와 공백만 남기고 구두점을 제거할 때 부정 문자 클래스를 사용할 수 있지만, 아포스트로피처럼 의미를 가진 기호도 함께 사라질 수 있다.

- **`[^\s\w]`**: 공백 문자와 단어 문자를 제외한 한 문자와 일치
- **빈 치환 문자열 `''`**: 일치한 문자를 다른 내용으로 바꾸지 않고 삭제

출력에서 구두점과 이모티콘이 제거되는 동시에 `Let's`가 `Lets`로 변하고 끝에 공백이 남는지 확인한다. 제거 규칙은 분석 목적과 보존해야 할 표현을 기준으로 결정한다.


In [27]:
text = "Hello, welcome to NLP! Let's clean text :)"

cleaned_text = re.sub(r'[^\s\w]', '', text)

print('원문:', text)

print('정제:', cleaned_text)

원문: Hello, welcome to NLP! Let's clean text :)
정제: Hello welcome to NLP Lets clean text 
